# Parallel Task API

Parallel's [Task API](https://docs.parallel.ai/task-api/overview) runs research-grade tasks across a tiered processor menu (`lite` → `ultra`, plus matching `-fast` variants).

`langchain-parallel` exposes the Task API as four LangChain surfaces:

| Surface | Shape | Use it for |
|---|---|---|
| [`ParallelTaskRunTool`](https://python.langchain.com/api_reference/parallel_web/tasks/langchain_parallel.tasks.ParallelTaskRunTool.html) | `BaseTool` | Single Task, callable by an LLM mid-conversation |
| [`ParallelDeepResearch`](https://python.langchain.com/api_reference/parallel_web/tasks/langchain_parallel.tasks.ParallelDeepResearch.html) | `Runnable` | Single deep-research task with structured output |
| [`ParallelTaskGroup`](https://python.langchain.com/api_reference/parallel_web/tasks/langchain_parallel.tasks.ParallelTaskGroup.html) | Plain class | Low-level batch primitive |
| [`ParallelEnrichment`](https://python.langchain.com/api_reference/parallel_web/tasks/langchain_parallel.tasks.ParallelEnrichment.html) | `Runnable` | Typed batch enrichment with pydantic schemas |

**All four default to a `-fast` processor variant** (`lite-fast` / `pro-fast` / `lite-fast` / `core-fast` respectively) — 2-5x faster than the corresponding non-fast tier at similar accuracy. Drop the `-fast` suffix when latency is less of a concern than maximum quality.

## Setup

In [ ]:
%pip install --quiet -U langchain-parallel

In [ ]:
import getpass
import os

if not os.environ.get("PARALLEL_API_KEY"):
    os.environ["PARALLEL_API_KEY"] = getpass.getpass("Parallel API key:\n")


## `ParallelTaskRunTool` — single Task with citations

`ParallelTaskRunTool` is an agent-callable `BaseTool`. It runs one Task synchronously and returns the structured `output`, per-field `basis` citations, and the `run_id`.

In [ ]:
from langchain_parallel import ParallelTaskRunTool

tool = ParallelTaskRunTool()  # default processor="lite-fast"
result = tool.invoke({"input": "Who founded SpaceX, in one sentence?"})
print(result["output"]["content"] if isinstance(result["output"], dict) else result["output"])
print("run_id:", result["run"]["run_id"])


The `output` shape depends on whether you pass a `task_output_schema`. With no schema (above), `result["output"]` is the raw text answer (or a dict). With a pydantic schema, you get a parsed instance and per-field citations:

In [ ]:
from pydantic import BaseModel, Field

class FounderFact(BaseModel):
    founder: str = Field(description="Full name of the founder")
    year: int = Field(description="Year the company was founded")

structured = ParallelTaskRunTool(task_output_schema=FounderFact)
res = structured.invoke({"input": "Who founded SpaceX and in what year?"})
print(res["output"])
# `res["basis"]` carries per-field citations, reasoning, confidence
print("basis fields:", [b.get("field") for b in res.get("basis", [])])


### `parse_basis()` — citations + low-confidence fields, in one call

Every consumer that cares about confidence ends up writing the same ~30 lines to walk a result and pull out citations + low-confidence fields + the `interaction_id` for chaining. `parse_basis()` does that for you.

In [ ]:
from langchain_parallel import parse_basis

parsed = parse_basis(res)
print("citations per field:", {f: len(c) for f, c in parsed["citations_by_field"].items()})
print("low-confidence fields:", parsed["low_confidence_fields"])
print("interaction_id:", parsed["interaction_id"])


## `ParallelDeepResearch` — single deep-research task

`ParallelDeepResearch` is a `Runnable`. It defaults to `pro-fast` (the `-fast` variant of "Exploratory web research"). For the most thorough reports, pass `processor="ultra"`.

In [ ]:
from langchain_parallel import ParallelDeepResearch

research = ParallelDeepResearch()  # default processor="pro-fast"
result = research.invoke("In one sentence, what is the capital of France?")
print(result["output"]["content"] if isinstance(result["output"], dict) else result["output"])


For typed deep research, pass an `output_schema`:

In [ ]:
class CityFact(BaseModel):
    capital: str
    population_millions: float

typed = ParallelDeepResearch(output_schema=CityFact)
out = typed.invoke("Capital and population (in millions) of France?")
print(out["output"])


## `ParallelTaskGroup` — low-level batch primitive

`ParallelTaskGroup` creates a Task Group, fans out runs, and collects results. Use it directly when you need fine-grained control over the batch envelope; otherwise prefer `ParallelEnrichment` (below) for typed bulk runs.

In [ ]:
from langchain_parallel import ParallelTaskGroup

group = ParallelTaskGroup()  # default processor="lite-fast"
results = group.run([
    "Founder of Anthropic? One sentence.",
    "Founder of OpenAI? One sentence.",
])
for r in results:
    out = r["output"]
    print(out["content"] if isinstance(out, dict) else out)


## `ParallelEnrichment` — typed batch enrichment

`ParallelEnrichment` wraps `ParallelTaskGroup` with a `default_task_spec` built from your input/output pydantic schemas. It coerces pydantic instances → dicts, fans out the batch, and returns results in input order.

In [ ]:
from langchain_parallel import ParallelEnrichment

class CompanyInput(BaseModel):
    company: str = Field(description="Company name")

class CompanyOutput(BaseModel):
    headquarters: str
    founding_year: int

enricher = ParallelEnrichment(
    input_schema=CompanyInput,
    output_schema=CompanyOutput,
    # default processor="core-fast" — pass "core" or "pro" for higher accuracy
)
results = enricher.invoke([
    CompanyInput(company="Anthropic"),
    {"company": "OpenAI"},
])
for r in results:
    print(r["output"]["content"])


## `build_task_spec` helper

`build_task_spec` accepts pydantic classes, raw JSON-schema dicts, or text descriptions and returns a TaskSpec dict ready for `client.task_run.create` or `add_runs(default_task_spec=...)`. Useful when you want full control of the run envelope.

In [ ]:
from langchain_parallel import build_task_spec

spec = build_task_spec(input_schema=CompanyInput, output_schema=CompanyOutput)
print(list(spec.keys()))
print(spec["output_schema"]["type"])  # "json"


## BYOMCP — bring your own MCP server

Both `ParallelTaskRunTool` and `ParallelDeepResearch` accept `mcp_servers=[McpServer(...)]` to expose your own Streamable-HTTP MCP endpoints to the run. We construct the spec without calling here so the notebook smoke stays self-contained.

In [ ]:
from langchain_parallel import McpServer

mcp = McpServer(
    name="my_internal_mcp",
    url="https://example.com/mcp",
    headers={"Authorization": "Bearer ..."},
)
print(mcp.model_dump())
# Pass it as: ParallelTaskRunTool(mcp_servers=[mcp]).invoke({"input": "..."})


## Webhook signature verification

Long-running Tasks can deliver results via webhook. Verify the signature with `verify_webhook` (Standard Webhooks scheme: HMAC-SHA256 over `<webhook-id>.<webhook-timestamp>.<body>`, base64-encoded, `v1,<sig>` format).

In [ ]:
import base64
import hashlib
import hmac
import time

from langchain_parallel import verify_webhook

# Simulate what the Parallel webhook delivery looks like.
secret = "test_secret"
body = b'{"run_id":"trun_abc"}'
webhook_id = "msg_demo"
ts = str(int(time.time()))
signed = f"{webhook_id}.{ts}.{body.decode()}"
sig = base64.b64encode(
    hmac.new(secret.encode(), signed.encode(), hashlib.sha256).digest()
).decode()

assert verify_webhook(
    body,
    webhook_id=webhook_id,
    webhook_timestamp=ts,
    webhook_signature=f"v1,{sig}",
    secret=secret,
)
print("signature verified")


## API reference

- [Task API guides](https://docs.parallel.ai/task-api/overview)
- [Choose a processor](https://docs.parallel.ai/task-api/guides/choose-a-processor)
- [Webhook setup](https://docs.parallel.ai/resources/webhook-setup)